In [2]:
import networkx as nx
import csv
import json
from tqdm import tqdm
import numpy as np
import matplotlib.pyplot as plt
from graph import Graph

In [3]:
all_files = {}

all_files['nodes_all'] = "2025_Problem_D_Data/nodes_all.csv"
all_files['edges_all'] = "2025_Problem_D_Data/edges_all.csv"
all_files['nodes_drive'] = "2025_Problem_D_Data/nodes_drive.csv"
all_files['edges_drive'] = "2025_Problem_D_Data/edges_drive.csv"
all_files['Bus_Stops'] = "2025_Problem_D_Data/Bus_Stops.csv"
all_files['SHA'] = "2025_Problem_D_Data/MDOT_SHA_Annual_Average_Daily_Traffic_Baltimore.csv"
all_files['Cache'] = "saved_graphs/network_graph.pkl"

In [4]:
G = Graph(all_files['nodes_all'], all_files['edges_all'], all_files['Bus_Stops'], all_files['Cache'])

Graph loaded from saved_graphs/network_graph.pkl


In [5]:
metrics = G.calculate_network_metrics()

for key, value in metrics.items():
    print(f"{key}: {value}")

total_nodes: 220226
total_edges: 553307
average_node_degree: 5.0249016918983225
average_in_degree: 2.5124508459491612
average_out_degree: 2.5124508459491612
max_in_degree: 7
max_out_degree: 10
average_clustering_coefficient: 0.03916689972670755
density: 1.1408563269152737e-05
diameter: inf
average_shortest_path_length: inf
assortativity: 0.052231174696880774
transitivity: 0.050154004596298166
strongly_connected_components: 162
weakly_connected_components: 1


In [6]:
def visualize_network_metrics(G):
    """Create bar plot of network metrics"""
    metrics = {
        'Total Nodes': G.number_of_nodes(),
        'Total Edges': G.number_of_edges(),
        'Avg Node Degree': np.mean([d for n, d in G.degree()]),
        'Avg In-Degree': np.mean([d for n, d in G.in_degree()]),
        'Avg Out-Degree': np.mean([d for n, d in G.out_degree()]),
    }
    
    plt.figure(figsize=(10, 6))
    plt.bar(metrics.keys(), metrics.values())
    plt.title('Transportation Network Metrics')
    plt.xticks(rotation=45, ha='right')
    plt.tight_layout()
    plt.savefig('figures/network_metrics.png')
    plt.close()

def plot_degree_distribution(G):
    """Plot degree distribution of the graph"""
    degrees = [d for n, d in G.degree()]
    plt.figure(figsize=(10, 6))
    plt.hist(degrees, bins=20, edgecolor='black')
    plt.title('Node Degree Distribution')
    plt.xlabel('Degree')
    plt.ylabel('Number of Nodes')
    plt.tight_layout()
    plt.savefig('figures/degree_distribution.png')
    plt.close()

def visualize_network_structure(G):
    """Create a simplified visualization of network structure"""
    plt.figure(figsize=(12, 8))
    pos = nx.spring_layout(G, k=0.5, iterations=50)
    nx.draw(G, pos, node_size=10, node_color='blue', 
            edge_color='gray', alpha=0.2, with_labels=False)
    plt.title('Transportation Network Structure')
    plt.tight_layout()
    plt.savefig('figures/network_structure.png')
    plt.close()

In [7]:
highway = {}

highway['others'] = 0

for edge in G.graph.edges:
    
    if (G.graph.edges[edge]['highway'][0] == '['):
        highway['others'] += 1
    else:
        if G.graph.edges[edge]['highway'] not in highway:
            highway[G.graph.edges[edge]['highway']] = 1
        else:
            highway[G.graph.edges[edge]['highway']] += 1

In [8]:
for key, value in highway.items():
    if (value < 100):
        highway['others'] += value

for key in list(highway.keys()):
    if (highway[key] < 100):
        del highway[key]

In [9]:
for key, value in highway.items():
    print(f"{key}: {value}")

others: 6531
motorway: 518
motorway_link: 940
residential: 115897
primary: 15591
track: 5127
tertiary: 28030
service: 224855
secondary: 19585
path: 16088
unclassified: 10116
secondary_link: 354
primary_link: 648
footway: 104112
trunk: 1388
trunk_link: 133
tertiary_link: 134
pedestrian: 770
steps: 458
cycleway: 2032


In [10]:
def visualize_highway_distribution(highway): 
    def autopct_format(pct):
        return f'{pct:.1f}%' if pct >= 2 else ''
    
    plt.figure(figsize=(12, 8))
    colors = plt.cm.tab20c.colors
    patches, texts, autotexts = plt.pie(
        highway.values(),
        labels=None,  
        autopct=autopct_format, 
        startangle=140,
        colors=colors[:len(highway)]
    )

    plt.title('Distribution of Highway Types')

    plt.legend(
        patches,
        [f"{key} ({value})" for key, value in highway.items()],
        loc='center right',  
        title="Highway Types",
        fontsize='small'
    )   


    # Save the figure
    plt.axis('equal') 
    plt.tight_layout()
    plt.savefig('figures/highway_distribution.png')
    plt.close()

In [11]:
def plot_aadt_distribution(G):
    """Plot AADT distribution of the graph nodes"""
    aadt_values = [data['average_aadt'] for _, data in G.graph.nodes(data=True) if 'average_aadt' in data and data['average_aadt'] > 0]
    
    plt.figure(figsize=(10, 6))
    plt.hist(aadt_values, bins=20, edgecolor='black')
    plt.title('AADT Distribution of Nodes')
    plt.xlabel('AADT')
    plt.ylabel('Number of Nodes')
    plt.tight_layout()
    plt.savefig('figures/aadt_distribution.png')
    plt.close()
    
plot_aadt_distribution(G)

In [12]:
import matplotlib.pyplot as plt
import seaborn as sns

def plot_aadt_boxplot(G):
    # Extract AADT values
    aadt_values = [data['average_aadt'] for _, data in G.graph.nodes(data=True) if 'average_aadt' in data and data['average_aadt'] > 0]
    
    # Initialize the plot
    plt.figure(figsize=(12, 6))
    sns.set_theme(style="whitegrid")

    # Create the box plot
    sns.boxplot(
        x=aadt_values,
        color="skyblue",
        width=0.6,
        flierprops={"marker": "o", "color": "red", "alpha": 0.6},  # Style outliers
        boxprops={"edgecolor": "blue", "linewidth": 1.5},
        whiskerprops={"linewidth": 1.5},
        capprops={"linewidth": 1.5},
        medianprops={"color": "darkred", "linewidth": 2}
    )
    
    # Add labels and title
    plt.title('AADT Box Plot of Nodes', fontsize=16, fontweight='bold')
    plt.xlabel('AADT', fontsize=14)
    plt.xticks(fontsize=12)
    plt.yticks([])  # Hide y-axis ticks since there's no categorical grouping
    plt.tight_layout()

    # Save and close the figure
    plt.savefig('figures/aadt_boxplot.png', dpi=300)
    plt.close()


In [13]:
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

def plot_geo_aadt_heatmap(G, grid_resolution=100):
    """Create a geographically accurate AADT heatmap"""
    # Extract node coordinates and AADT values
    coords = []
    aadt_values = []
    
    for node, data in G.graph.nodes(data=True):
        if 'geometry' in data and data['geometry'].startswith("POINT"):
            coordinates = [float(coord) for coord in data['geometry'].strip("POINT ()").split()]
        if data['average_aadt'] > 0:
            coords.append(coordinates)
            aadt_values.append(data['average_aadt'])
    
    # Convert to numpy arrays
    coords = np.array(coords)
    aadt_values = np.array(aadt_values)
    
    # Create 2D histogram
    heatmap, xedges, yedges = np.histogram2d(
        coords[:, 0], coords[:, 1], 
        bins=grid_resolution, 
        weights=aadt_values
    )
    
    # Plot
    plt.figure(figsize=(15, 10))
    plt.imshow(heatmap.T, origin='lower', aspect='auto', 
               extent=[xedges[0], xedges[-1], yedges[0], yedges[-1]], 
               cmap='YlOrRd')
    plt.colorbar(label='Average AADT')
    plt.title('Geographical AADT Heatmap', fontsize=16)
    plt.xlabel('Longitude', fontsize=12)
    plt.ylabel('Latitude', fontsize=12)
    plt.tight_layout()
    plt.savefig('figures/geo_aadt_heatmap.png', dpi=300)
    plt.close()

In [14]:
def finding_top_twenty_bottle_necks(G):
    # Sort nodes by AADT
    nodes = sorted(G.graph.nodes(data=True), key=lambda x: x[1].get('average_aadt', 0), reverse=True)
    
    top_twenty = nodes[:20]
    
    return top_twenty
    

In [15]:
top_twenty = finding_top_twenty_bottle_necks(G) 

In [16]:
print("Top 20 Bottlenecks:")
for i, (node, data) in enumerate(top_twenty):
    print(f"{i+1}. Node {node} - AADT: {data.get('average_aadt', 0)} - Location: {data.get('geometry', 'N/A')}")

Top 20 Bottlenecks:
1. Node 664286830 - AADT: 1117580.9999999998 - Location: POINT (-76.6694678 39.418665)
2. Node 9768381905 - AADT: 1117580.9999999998 - Location: POINT (-76.6693374 39.4187001)
3. Node 37674488 - AADT: 895666.8 - Location: POINT (-76.4274684 39.3801998)
4. Node 37675106 - AADT: 895666.8 - Location: POINT (-76.4276421 39.3802827)
5. Node 37592041 - AADT: 762351.6 - Location: POINT (-76.5123688 39.2999509)
6. Node 37561398 - AADT: 729350.3999999996 - Location: POINT (-76.7572236 39.4035515)
7. Node 37572454 - AADT: 697762.8 - Location: POINT (-76.630493 39.4134464)
8. Node 37635415 - AADT: 632645.2000000001 - Location: POINT (-76.6410852 39.4748309)
9. Node 37561393 - AADT: 598716.3999999997 - Location: POINT (-76.7490616 39.397417)
10. Node 37572259 - AADT: 568100.0000000001 - Location: POINT (-76.6274636 39.3824953)
11. Node 37643009 - AADT: 563267.6000000001 - Location: POINT (-76.6165695 39.4204909)
12. Node 49473594 - AADT: 562638.3 - Location: POINT (-76.6916947 

In [17]:
top_twenty_bottlenecks = [node for node, _ in top_twenty]

G.to_geojson_node(top_twenty_bottlenecks)

GeoJSON file created.


{'type': 'FeatureCollection',
 'features': [{'type': 'Feature',
   'properties': {'node_id': '664286830'},
   'geometry': {'type': 'Point', 'coordinates': [-76.6694678, 39.418665]}},
  {'type': 'Feature',
   'properties': {'node_id': '9768381905'},
   'geometry': {'type': 'Point', 'coordinates': [-76.6693374, 39.4187001]}},
  {'type': 'Feature',
   'properties': {'node_id': '37674488'},
   'geometry': {'type': 'Point', 'coordinates': [-76.4274684, 39.3801998]}},
  {'type': 'Feature',
   'properties': {'node_id': '37675106'},
   'geometry': {'type': 'Point', 'coordinates': [-76.4276421, 39.3802827]}},
  {'type': 'Feature',
   'properties': {'node_id': '37592041'},
   'geometry': {'type': 'Point', 'coordinates': [-76.5123688, 39.2999509]}},
  {'type': 'Feature',
   'properties': {'node_id': '37561398'},
   'geometry': {'type': 'Point', 'coordinates': [-76.7572236, 39.4035515]}},
  {'type': 'Feature',
   'properties': {'node_id': '37572454'},
   'geometry': {'type': 'Point', 'coordinates'